# TEBD Helper APIs on a Physical Spin Chain

This notebook walks through the compact helper layer for finite-chain TEBD on physical spin-1/2 sites.

We will build up the API in the same order you are likely to meet it in practice:

1. `local_gates_from_hamiltonians`
2. `tebd_evolution_from_hamiltonians`
3. `tebd_strang_schedule`
4. `tebd_strang_evolution`

The point is to show what each helper does and how much manual work it removes.


In [ ]:
using LinearAlgebra
using Plots
using ITensors
using ITensorMPS
using MPSToolkit

default(size=(960, 680), linewidth=2, markersize=4, legend=:best)

spins = spinhalf_matrices()

function xxz_bond_hamiltonian(; Jxy::Real=1.0, Delta::Real=1.1)
    return Jxy * kron(spins.Sx, spins.Sx) +
           Jxy * kron(spins.Sy, spins.Sy) +
           Delta * kron(spins.Sz, spins.Sz)
end

function local_sz_profile(psi::MPS)
    return real.(expect(psi, "Sz"))
end


## 1. `local_gates_from_hamiltonians`

This is the lowest-level helper in the stack.
It turns dense local Hamiltonian data into dense local gates.

It accepts:

- one dense Hamiltonian matrix
- a vector of dense Hamiltonians
- a callable Hamiltonian provider


In [ ]:
dt = 0.08
hbond = xxz_bond_hamiltonian()

single_gate = local_gates_from_hamiltonians(hbond, dt)
vector_gates = local_gates_from_hamiltonians([hbond, 0.5 .* hbond], dt)
callable_gates = local_gates_from_hamiltonians((bond, index) -> (0.2 * index) .* hbond, dt)

println("single gate size      = ", size(single_gate))
println("vector gate sizes     = ", [size(g) for g in vector_gates])
println("callable gate (1, 1)  = ", size(callable_gates(1, 1)))
println("callable gate (2, 2)  = ", size(callable_gates(2, 2)))


## 2. `tebd_evolution_from_hamiltonians`

This helper takes the next step: instead of only giving you gates, it builds a `LocalGateEvolution` directly.
That is useful when you already know the schedule you want but do not want to manually exponentiate the local terms.


In [ ]:
nsites = 6
sites = siteinds("S=1/2", nsites)
psi_manual = MPS(sites, n -> isodd(n) ? "Up" : "Dn")
psi_helper = copy(psi_manual)

schedule = [1, 2, 3, 4, 5, 5, 4, 3, 2, 1]
gates = fill(local_gates_from_hamiltonians(hbond, dt), length(schedule))
manual_evolution = LocalGateEvolution(gates, dt; schedule=schedule, maxdim=32, cutoff=1e-10)
helper_evolution = tebd_evolution_from_hamiltonians(fill(hbond, length(schedule)), dt; schedule=schedule, maxdim=32, cutoff=1e-10)

evolve!(psi_manual, manual_evolution)
evolve!(psi_helper, helper_evolution)

println("manual/helper overlap = ", abs(inner(psi_manual, psi_helper)))
bar(1:nsites, local_sz_profile(psi_helper); xlabel="site", ylabel="⟨Sz⟩", title="After one scheduled helper-built evolution", label="helper evolution")


## 3. `tebd_strang_schedule`

For nearest-neighbor open-chain TEBD, the package can build the standard odd-even-odd Strang schedule for you.
It also returns the per-step weights that tell you where the half steps and full steps occur.


In [ ]:
schedule_strang, weights_strang = tebd_strang_schedule(nsites)
println("Strang schedule = ", schedule_strang)
println("Strang weights  = ", weights_strang)
plot(1:length(weights_strang), weights_strang; marker=:circle, xlabel="schedule index", ylabel="weight", title="Odd-even-odd Strang weights", label="weight")


## 4. `tebd_strang_evolution`

This is the most compact helper in the stack.
You provide a local Hamiltonian builder, and the helper creates the odd-even-odd schedule, applies the Strang weights, and returns a `LocalGateEvolution` ready for `evolve!`.


In [ ]:
psi_strang = MPS(sites, n -> isodd(n) ? "Up" : "Dn")
strang_evolution = tebd_strang_evolution(
    nsites,
    dt;
    local_hamiltonian=(bond, weight) -> weight * xxz_bond_hamiltonian(Jxy=1.0, Delta=0.9),
    maxdim=32,
    cutoff=1e-10,
)

profiles = [local_sz_profile(psi_strang)]
for _ in 1:8
    evolve!(psi_strang, strang_evolution)
    push!(profiles, local_sz_profile(psi_strang))
end

heatmap(0:8, 1:nsites, reduce(hcat, profiles); xlabel="Strang step", ylabel="site", title="Packed Strang helper workflow", colorbar_title="⟨Sz⟩")


## 5. Manual vs helper route

The helper APIs are meant to compress boilerplate, not hide what is happening.
A good sanity check is that the packed helper route reproduces the same result as the more manual route when the local data and schedule agree.


In [ ]:
psi_a = MPS(sites, n -> isodd(n) ? "Up" : "Dn")
psi_b = copy(psi_a)

schedule_check, weights_check = tebd_strang_schedule(nsites)
hamiltonians = [weight * xxz_bond_hamiltonian(Jxy=1.0, Delta=0.9) for weight in weights_check]
manual = tebd_evolution_from_hamiltonians(hamiltonians, dt; schedule=schedule_check, maxdim=32, cutoff=1e-10)
packed = tebd_strang_evolution(nsites, dt; local_hamiltonian=(bond, weight) -> weight * xxz_bond_hamiltonian(Jxy=1.0, Delta=0.9), maxdim=32, cutoff=1e-10)

evolve!(psi_a, manual)
evolve!(psi_b, packed)
println("manual vs packed overlap = ", abs(inner(psi_a, psi_b)))
